# Validate a circuit: switch or readout?

A toy model with known answers: one feature causes the answer (a **switch**), two light up but do nothing (**readouts**), and two back each other up (**backups**). `validate` should tell them apart with ablation, steering and a counterfactual patch, each against random and matched-norm controls.

In [1]:
from circuitkit.validate import logit_diff, validate
from circuitkit.validate.toy import demo, demo_nodes

backend = demo()
metric = logit_diff(backend.token_id(" Austin"), backend.token_id(" Sacramento"), " Austin", " Sacramento")
report = validate(
    backend, "dallas", demo_nodes(backend), metric=metric, counterfactual="oakland", n_controls=20, seed=0
)
for g in report.groups:
    print(f"{g.name:>10}  {g.verdict.value}")

     top-5  switch
  L5/100@5  switch
  L7/200@5  readout
  L8/201@5  readout
  L4/300@5  backup
  L6/301@5  backup


Each verdict comes with the numbers behind it:

In [2]:
for reason in report.groups[1].reasons:
    print("-", reason)

- ablation moves the metric by -4.000 (43% of scale; needs 20%); p=0.048 vs random, p=0.048 vs matched_norm
- steering moves it by +8.000 (86%); p=0.048 vs random, p=0.048 vs matched_norm
- counterfactual patch closes 43% of the gap (needs 30%); p=0.048 vs random, p=0.048 vs matched_norm


The same report as a self-contained HTML page:

In [3]:
from circuitkit.report import render_validate, save_html

save_html(render_validate(report), "reports/validate_toy.html")

PosixPath('reports/validate_toy.html')